In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

labels_mcc = sorted(set(y_test_m))
cm_mcc     = confusion_matrix(y_test_m, y_pred_m, labels=labels_mcc)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_mcc, annot=True, fmt='d', cmap='Greens',
    xticklabels=labels_mcc, yticklabels=labels_mcc
)
plt.title('Matrice de confusion — Random Forest — MCC5-THU Gearbox')
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.tight_layout()
plt.savefig('../data/processed/confusion_matrix_mcc5.png', dpi=150)
plt.show()

# Sauvegarde
np.save('../data/processed/X_mcc.npy', X_mcc)
with open('../data/processed/y_mcc.pkl', 'wb') as f:
    pickle.dump(y_mcc, f)
with open('../data/processed/meta_mcc.pkl', 'wb') as f:
    pickle.dump(meta_mcc, f)

print(f"✅ Sauvegardé — X_mcc : {X_mcc.shape}")

In [2]:
from pathlib import Path
import shutil

cache_dir = Path("data/cache")
if cache_dir.exists():
    # Supprimer tous les caches ou seulement celui de MCC5-THU
    for f in cache_dir.glob("*.pkl"):
        f.unlink()
    print("Caches supprimés")

Caches supprimés


In [5]:
import re

# Grouper par fichier source
file_groups = defaultdict(list)
for i, m in enumerate(meta_mcc):
    file_groups[m['unit_id']].append(i)

# Identifier la condition de chaque fichier
def get_condition(uid):
    rpm_m = re.search(r'(\d{3,4})rpm', uid)
    nm_m  = re.search(r'(\d+)Nm',     uid)
    rpm   = rpm_m.group(1) if rpm_m else '2000'
    nm    = nm_m.group(1)  if nm_m  else '10'
    return f"{rpm}rpm_{nm}Nm"

# Split : train = 1000rpm + 2000rpm | test = 3000rpm
train_files = [fid for fid in file_groups
               if '3000rpm' not in get_condition(fid)]
test_files  = [fid for fid in file_groups
               if '3000rpm' in get_condition(fid)]

train_idx = [i for fid in train_files for i in file_groups[fid]]
test_idx  = [i for fid in test_files  for i in file_groups[fid]]

X_train = X_mcc[train_idx]
X_test  = X_mcc[test_idx]
y_train = [y_mcc[i] for i in train_idx]
y_test  = [y_mcc[i] for i in test_idx]

print(f"Train : {len(train_files)} fichiers → {len(X_train)} fenêtres")
print(f"Test  : {len(test_files)} fichiers  → {len(X_test)} fenêtres")
print(f"Classes test : {sorted(set(y_test))}")

Train : 80 fichiers → 119840 fenêtres
Test  : 34 fichiers  → 50932 fenêtres
Classes test : ['dent_cassee', 'dent_manquante', 'fissure', 'mixte_externe', 'mixte_interne', 'pitting', 'sain', 'usure']


In [6]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight

# Encodage des labels
le = LabelEncoder()
le.fit(y_train + y_test)
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)

# Poids des échantillons pour équilibrer les classes
sample_weights = compute_sample_weight('balanced', y_train_enc)

# XGBoost
xgb = XGBClassifier(
    n_estimators      = 300,
    max_depth         = 6,
    learning_rate     = 0.05,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    use_label_encoder = False,
    eval_metric       = 'mlogloss',
    random_state      = 42,
    n_jobs            = -1
)

print("Entraînement XGBoost...")
xgb.fit(X_train, y_train_enc, sample_weight=sample_weights)

y_pred_enc = xgb.predict(X_test)
y_pred     = le.inverse_transform(y_pred_enc)
acc        = accuracy_score(y_test, y_pred)

print(f"\nAccuracy XGBoost (features enrichies) : {acc*100:.2f}%")
print(classification_report(y_test, y_pred))

Entraînement XGBoost...


c:\Users\HP PRO\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:05:45] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Accuracy XGBoost (features enrichies) : 41.03%
                precision    recall  f1-score   support

   dent_cassee       0.32      0.24      0.28      5992
dent_manquante       0.31      0.06      0.10      5992
       fissure       0.64      0.20      0.31      5992
 mixte_externe       0.61      0.34      0.43      7490
 mixte_interne       0.38      0.59      0.46      7490
       pitting       0.24      0.20      0.22      4494
          sain       0.67      0.72      0.70      5992
         usure       0.33      0.77      0.46      7490

      accuracy                           0.41     50932
     macro avg       0.44      0.39      0.37     50932
  weighted avg       0.44      0.41      0.38     50932



In [4]:
import sys
sys.path.append('../')

from backend.ml.adapters.dataset_adapter import DatasetAdapter
from backend.ml.features.feature_extractor import (
    extract_features_mcc5_v2, extract_rpm_from_uid
)
from collections import Counter, defaultdict
import numpy as np, pickle, re

adapter_mcc = DatasetAdapter('../configs/datasets/mcc5_thu.yaml')

signals_mcc = adapter_mcc.load(
    '../data/raw/mcc5_thu/mcc5_thu.zip',
    max_files=15,
    use_cache=True
)
print(f"Signaux : {len(signals_mcc)}")

  Cache trouvé (8015f7fe4aef.pkl) — chargement rapide...
Signaux : 114


In [6]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder


# Grouper par fichier source
file_groups = defaultdict(list)
file_label = {}
for i, sig in enumerate(signals_mcc):
    fid = sig.unit_id
    file_groups[fid].append(i)
    file_label[fid] = sig.label

all_fids = list(file_groups.keys())
all_flabels = [file_label[f] for f in all_fids]

# Split stratifié 80/20
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx_f, test_idx_f = next(sss.split(all_fids, all_flabels))

train_files = [all_fids[i] for i in train_idx_f]
test_files  = [all_fids[i] for i in test_idx_f]

# Construire les indices (sig_idx, win_idx) pour train et test
train_indices = []
test_indices = []
for sig_idx, sig in enumerate(signals_mcc):
    fid = sig.unit_id
    for win_idx in range(sig.signals.shape[0]):
        if fid in train_files:
            train_indices.append((sig_idx, win_idx))
        elif fid in test_files:
            test_indices.append((sig_idx, win_idx))

print(f"Train : {len(train_indices)} fenêtres ({len(train_files)} fichiers)")
print(f"Test  : {len(test_indices)} fenêtres ({len(test_files)} fichiers)")

# Encoder les labels
le = LabelEncoder()
all_labels = [sig.label for sig in signals_mcc]
le.fit(all_labels)
print(f"Classes ({len(le.classes_)}) : {list(le.classes_)}")

Train : 136318 fenêtres (91 fichiers)
Test  : 34454 fenêtres (23 fichiers)
Classes (8) : [np.str_('dent_cassee'), np.str_('dent_manquante'), np.str_('fissure'), np.str_('mixte_externe'), np.str_('mixte_interne'), np.str_('pitting'), np.str_('sain'), np.str_('usure')]


In [8]:
from torch.utils.data import DataLoader
from backend.ml.models.cnn_gear import LazyGearDataset  # nouvelle classe


BATCH_SIZE = 256   # réduire à 128 si mémoire GPU faible

train_dataset = LazyGearDataset(signals_mcc, le, train_indices)
test_dataset  = LazyGearDataset(signals_mcc, le, test_indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Batches train : {len(train_loader)}")
print(f"Batches test  : {len(test_loader)}")

Batches train : 533
Batches test  : 135
